In [1]:
import os
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import seaborn as sns

# ==========================================
# 0. GLOBAL PLOTTING & ENVIRONMENT SETUP
# ==========================================
sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 11, 'figure.autolayout': True})

#pl.Config.set_tbl_rows(-1)             # shows all table rows globably
#pl.Config.set_tbl_cols(-1)             # shows all table columns globably
#pl.Config.set_fmt_str_lengths(100)     # increase character count visible globably
#pl.Config.restore_defaults()           # reset to default configuration

# Get current working directory (e.g., .\notebooks)
cwd = os.getcwd()

# Get parent folder to access (.\Data)
parent = os.path.dirname(cwd)

# Get file path to load into environment
file_path_EPTI_data = os.path.join(parent, "Data", "EPTI_CET_data.parquet")

print(f"Loading dataset from: {file_path_EPTI_data}")
df = pl.read_parquet(file_path_EPTI_data).with_columns(pl.col(["Price_profile", "Price_level", "Region",
                                                                "Municipality", "Group_Phase2", "Control_Price_Phase2"]).cast(pl.Categorical))

Loading dataset from: c:\Users\Marty\Documents\School\Merrimack College\8) DSE6311 Data Science Capstone\team_omega\Data\EPTI_CET_data.parquet


In [2]:
# Check for missing values
def what_percentage_missing(df):
    missing_percentage = (
        df.select((pl.all().is_null().sum() / pl.len()) * 100)
        .transpose(include_header=True, header_name="column", column_names=["pct_missing"])
        .sort("pct_missing", descending=True)
    )

    print("Percentage of missing values in each column:")

    with pl.Config(tbl_rows=-1):
        print(missing_percentage)

what_percentage_missing(df)

Percentage of missing values in each column:
shape: (18, 2)
┌──────────────────────────┬─────────────┐
│ column                   ┆ pct_missing │
│ ---                      ┆ ---         │
│ str                      ┆ f64         │
╞══════════════════════════╪═════════════╡
│ Price_profile            ┆ 88.006897   │
│ Price_level              ┆ 88.006897   │
│ Experiment_price_NOK_kWh ┆ 88.006897   │
│ ID                       ┆ 0.0         │
│ From                     ┆ 0.0         │
│ Date                     ┆ 0.0         │
│ Day_Of_Week              ┆ 0.0         │
│ Is_Weekend               ┆ 0.0         │
│ Hour                     ┆ 0.0         │
│ Demand_kWh               ┆ 0.0         │
│ Temperature              ┆ 0.0         │
│ Temperature24            ┆ 0.0         │
│ Temperature48            ┆ 0.0         │
│ Temperature72            ┆ 0.0         │
│ Municipality             ┆ 0.0         │
│ Region                   ┆ 0.0         │
│ Control_Price_Phase2     ┆ 0.0     

In [3]:
# Create a boolean expression checking for "Control"
is_control = pl.col("Group_Phase2").cast(pl.String).str.contains("Control")

# Summary DataFrame with counts and percentages
summary_df = (
    df.with_columns(
        pl.when(is_control)
        .then(pl.lit("Control"))
        .otherwise(pl.lit("Experiment"))
        .alias("Cohort")
    )
    .group_by("Cohort")
    .agg(pl.len().alias("count"))
    .with_columns(
        (pl.col("count") / pl.col("count").sum() * 100).round(2).alias("pct")
    )
)

print(summary_df)

shape: (2, 3)
┌────────────┬─────────┬───────┐
│ Cohort     ┆ count   ┆ pct   │
│ ---        ┆ ---     ┆ ---   │
│ str        ┆ u32     ┆ f64   │
╞════════════╪═════════╪═══════╡
│ Experiment ┆ 8638752 ┆ 70.93 │
│ Control    ┆ 3541248 ┆ 29.07 │
└────────────┴─────────┴───────┘


In [4]:
# Subset containing Ber_Control, Bo_Control, Control, Os_Control, Trom_Control, Trond_Control
df_control = df.filter(is_control)

# Subset containing everything else (Ber_1, Bo_1, H1, H2, NA, Os_1..6, Trom_1..2, Trond_1)
df_experiment = df.filter(~is_control)

In [5]:
# just the dataframe with 
what_percentage_missing(df_experiment)

Percentage of missing values in each column:
shape: (18, 2)
┌──────────────────────────┬─────────────┐
│ column                   ┆ pct_missing │
│ ---                      ┆ ---         │
│ str                      ┆ f64         │
╞══════════════════════════╪═════════════╡
│ Price_profile            ┆ 83.090613   │
│ Price_level              ┆ 83.090613   │
│ Experiment_price_NOK_kWh ┆ 83.090613   │
│ ID                       ┆ 0.0         │
│ From                     ┆ 0.0         │
│ Date                     ┆ 0.0         │
│ Day_Of_Week              ┆ 0.0         │
│ Is_Weekend               ┆ 0.0         │
│ Hour                     ┆ 0.0         │
│ Demand_kWh               ┆ 0.0         │
│ Temperature              ┆ 0.0         │
│ Temperature24            ┆ 0.0         │
│ Temperature48            ┆ 0.0         │
│ Temperature72            ┆ 0.0         │
│ Municipality             ┆ 0.0         │
│ Region                   ┆ 0.0         │
│ Control_Price_Phase2     ┆ 0.0     

In [ ]:
# 100% missing - as expected
#what_percentage_missing(df_control)

In [6]:
# How many and what proportion of those experiment days fall specifically on weekends
total_rows = df.height

stats = (
    df_experiment
    .filter(
        (pl.col("Is_Weekend") == 1) & 
        (pl.col("Experiment_price_NOK_kWh").is_not_null()) &
        (pl.col("Price_profile").is_not_null()) &
        (pl.col("Price_level").is_not_null())
    )
    .select(
        Match_Count = pl.len()
    )
    .with_columns(
        Percentage = (pl.col("Match_Count") / total_rows) * 100
    )
)

print(total_rows)
print(stats)

12180000
shape: (1, 2)
┌─────────────┬────────────┐
│ Match_Count ┆ Percentage │
│ ---         ┆ ---        │
│ u32         ┆ f64        │
╞═════════════╪════════════╡
│ 9813        ┆ 0.080567   │
└─────────────┴────────────┘


In [7]:
# Very few occurances - it is possible that some of these are just "bleed" from a Friday night midnight into Saturday
# We can not create an Is_Weekend variant of our proposed model, with these data. Bummer.
weekend_anomalies = (
    df
    .filter((pl.col("Is_Weekend") == 1) & (pl.col("Price_level").is_not_null()))
    .group_by(["Municipality", "Price_profile", "Hour"])
    .len()
    .sort("len", descending=True)
)

with pl.Config(tbl_rows=-1, tbl_cols=-1):
    print(weekend_anomalies)

shape: (33, 4)
┌──────────────┬───────────────┬──────┬──────┐
│ Municipality ┆ Price_profile ┆ Hour ┆ len  │
│ ---          ┆ ---           ┆ ---  ┆ ---  │
│ cat          ┆ cat           ┆ i32  ┆ u32  │
╞══════════════╪═══════════════╪══════╪══════╡
│ Oslo         ┆ P             ┆ 0    ┆ 1076 │
│ Tromsø       ┆ P             ┆ 0    ┆ 1060 │
│ Tromsø       ┆ B             ┆ 0    ┆ 1036 │
│ Bergen       ┆ B             ┆ 0    ┆ 807  │
│ Trondheim    ┆ B             ┆ 0    ┆ 804  │
│ Oslo         ┆ B             ┆ 0    ┆ 648  │
│ Oslo         ┆ A             ┆ 0    ┆ 454  │
│ Oslo         ┆ C             ┆ 0    ┆ 449  │
│ Bodø         ┆ B             ┆ 0    ┆ 442  │
│ Oslo         ┆ P0            ┆ 0    ┆ 436  │
│ Bærum        ┆ P             ┆ 0    ┆ 337  │
│ Bærum        ┆ B             ┆ 0    ┆ 192  │
│ Lillestrøm   ┆ P             ┆ 0    ┆ 161  │
│ Stavanger    ┆ B             ┆ 0    ┆ 156  │
│ Stavanger    ┆ P             ┆ 0    ┆ 156  │
│ Bergen       ┆ P             ┆ 0    ┆ 138  

In [8]:
# Check for duplicate rows
duplicate_rows = df.is_duplicated().sum()
print(f"Number of duplicate rows: {duplicate_rows}")

Number of duplicate rows: 0


In [9]:
# Check data types
print("Data types of each column:")
schema_df = pl.DataFrame({
    "Column": df.columns,
    "Data Type": [str(dtype) for dtype in df.dtypes]})

with pl.Config(tbl_rows=-1):
    print(schema_df)

Data types of each column:
shape: (18, 2)
┌──────────────────────────┬─────────────────────────────────┐
│ Column                   ┆ Data Type                       │
│ ---                      ┆ ---                             │
│ str                      ┆ str                             │
╞══════════════════════════╪═════════════════════════════════╡
│ ID                       ┆ String                          │
│ From                     ┆ Datetime(time_unit='ms', time_… │
│ Date                     ┆ Date                            │
│ Day_Of_Week              ┆ Int32                           │
│ Is_Weekend               ┆ Int32                           │
│ Hour                     ┆ Int32                           │
│ Demand_kWh               ┆ Float64                         │
│ Price_profile            ┆ Categorical                     │
│ Price_level              ┆ Categorical                     │
│ Experiment_price_NOK_kWh ┆ Float64                         │
│ Temperature

In [ ]:
# Calculate the target label (Y)


In [ ]:
# Audit and filter raw records following Hofmann & Lindberg's quality protocol:
# - Remove negative values, extreme spikes (>25 kWh), and anomalous households (mean > 9.9 kWh).
# - Identify and flag multi-day consecutive zero-kWh runs (vacant homes / meter dropouts).

print("Data Quality Audit")
initial_row_count = df.height
initial_households = df.select("ID").n_unique()

# Filter out invalid individual hourly readings (negative values, extreme spikes > 25 kWh)
df_audited = df.filter(
    (pl.col("Demand_kWh") >= 0.0) & 
    (pl.col("Demand_kWh") <= 25.0)
)
rows_after_hourly_filter = df_audited.height
print(f"Removed {initial_row_count - rows_after_hourly_filter} invalid hourly readings (negative or >25 kWh).")

# Exclude households whose overall mean consumption exceeds 9.9 kWh
household_means = (
    df_audited
    .group_by("ID")
    .agg(pl.col("Demand_kWh").mean().alias("Mean_Demand"))
)

valid_households = (
    household_means
    .filter(pl.col("Mean_Demand") <= 9.9)
    .select("ID")
)

df_audited = df_audited.join(valid_households, on="ID", how="inner")
rows_after_household_filter = df_audited.height
excluded_households = initial_households - df_audited.select("ID").n_unique()
print(f"Excluded {excluded_households} households with mean consumption > 9.9 kWh.")
print(f"Rows remaining after household-level filter: {rows_after_household_filter}")


print("--- Running Robust Consecutive Zero Audit ---")

# Ensure data is strictly sorted by ID and chronological Timestamp/Date-Hour
df_audited = df_audited.sort(["ID", "Date", "Hour"])

# Flag rows where demand is exactly zero
df_audited = df_audited.with_columns(
    Is_Zero=(pl.col("Demand_kWh") == 0.0)
)

print("test of audited")
#print(df_audited.head())
print(df_audited.select(pl.col("Is_Zero").not_().sum().alias("false_count")))
#print(df_audited)
# Only 742 instances found where Demand is 0 kWh 0.006% of total rows - we will keep these in the dataset

Data Quality Audit
Removed 0 invalid hourly readings (negative or >25 kWh).
Excluded 0 households with mean consumption > 9.9 kWh.
Rows remaining after household-level filter: 12180000
--- Running Robust Consecutive Zero Audit ---
test of audited
shape: (1, 1)
┌─────────────┐
│ false_count │
│ ---         │
│ u32         │
╞═════════════╡
│ 12179258    │
└─────────────┘
shape: (12_180_000, 19)
┌────────┬─────────────┬────────────┬─────────────┬───┬────────┬────────────┬────────────┬─────────┐
│ ID     ┆ From        ┆ Date       ┆ Day_Of_Week ┆ … ┆ Region ┆ Control_Pr ┆ Group_Phas ┆ Is_Zero │
│ ---    ┆ ---         ┆ ---        ┆ ---         ┆   ┆ ---    ┆ ice_Phase2 ┆ e2         ┆ ---     │
│ str    ┆ datetime[ms ┆ date       ┆ i32         ┆   ┆ cat    ┆ ---        ┆ ---        ┆ bool    │
│        ┆ , UTC]      ┆            ┆             ┆   ┆        ┆ cat        ┆ cat        ┆         │
╞════════╪═════════════╪════════════╪═════════════╪═══╪════════╪════════════╪════════════╪════════

In [36]:
# Group by Date, ID, Hour, and Temperature Bin from non-experiment weekdays.
df_processed = df_audited.with_columns(
    Temperature_Bin=((pl.col("Temperature24") / 5.0).round() * 5.0).alias("Temperature_Bin")
)

# Baseline lookup using non-experiment weekdays
baseline_lookup = (
    df_processed
    .filter(
        (pl.col("Experiment_price_NOK_kWh").is_null()) & 
        (pl.col("Is_Weekend") == 0)
    )
    .group_by(["ID", "Date", "Hour", "Temperature_Bin"])
    .agg(
        pl.col("Demand_kWh").mean().alias("Baseline_Demand_kWh")
    )
)

print(baseline_lookup)

shape: (7_364_678, 5)
┌──────────┬────────────┬──────┬─────────────────┬─────────────────────┐
│ ID       ┆ Date       ┆ Hour ┆ Temperature_Bin ┆ Baseline_Demand_kWh │
│ ---      ┆ ---        ┆ ---  ┆ ---             ┆ ---                 │
│ str      ┆ date       ┆ i32  ┆ f64             ┆ f64                 │
╞══════════╪════════════╪══════╪═════════════════╪═════════════════════╡
│ Exp_2908 ┆ 2021-03-09 ┆ 21   ┆ 0.0             ┆ 0.92                │
│ Exp_69   ┆ 2021-01-29 ┆ 10   ┆ -5.0            ┆ 5.155               │
│ Exp_4485 ┆ 2020-12-10 ┆ 19   ┆ 0.0             ┆ 1.25                │
│ Exp_5145 ┆ 2020-12-14 ┆ 13   ┆ 5.0             ┆ 8.026               │
│ Exp_2638 ┆ 2021-01-26 ┆ 6    ┆ -5.0            ┆ 7.93                │
│ …        ┆ …          ┆ …    ┆ …               ┆ …                   │
│ Exp_4347 ┆ 2020-12-24 ┆ 21   ┆ 0.0             ┆ 2.89                │
│ Exp_2704 ┆ 2020-12-02 ┆ 22   ┆ 0.0             ┆ 1.18                │
│ Exp_1870 ┆ 2020-12-11 ┆ 17 

In [44]:
# Join the counterfactuals with the baseline info
df_with_baseline = (
    df_processed
    .join(baseline_lookup, on=["ID", "Date", "Hour", "Temperature_Bin"], how="left")
    .with_columns(
        E_base_h=pl.col("Baseline_Demand_kWh").fill_null(0.0)
    )
)

print(df_with_baseline)

shape: (12_180_000, 26)
┌────────┬────────────┬────────────┬────────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ ID     ┆ From       ┆ Date       ┆ Day_Of_Wee ┆ … ┆ Multi_Day ┆ Temperatu ┆ Baseline_ ┆ E_base_h │
│ ---    ┆ ---        ┆ ---        ┆ k          ┆   ┆ _Zero_Fla ┆ re_Bin    ┆ Demand_kW ┆ ---      │
│ str    ┆ datetime[m ┆ date       ┆ ---        ┆   ┆ g         ┆ ---       ┆ h         ┆ f64      │
│        ┆ s, UTC]    ┆            ┆ i32        ┆   ┆ ---       ┆ f64       ┆ ---       ┆          │
│        ┆            ┆            ┆            ┆   ┆ i32       ┆           ┆ f64       ┆          │
╞════════╪════════════╪════════════╪════════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ Exp_1  ┆ 2020-12-01 ┆ 2020-12-01 ┆ 3          ┆ … ┆ null      ┆ 5.0       ┆ 1.891     ┆ 1.891    │
│        ┆ 00:00:00   ┆            ┆            ┆   ┆           ┆           ┆           ┆          │
│        ┆ UTC        ┆            ┆            ┆   ┆           ┆  

In [41]:
epsilon = 1e-6  # Smoothing constant

# Aggregate peak-window totals for experiment days
fpi_aggregated = (
    df_with_baseline
    .filter(pl.col("Experiment_price_NOK_kWh").is_not_null())
    .group_by(["ID", "Date"])
    .agg([
        pl.col("Demand_kWh").sum().alias("E_actual_peak"),
        pl.col("E_base_h").sum().alias("E_base_peak"),
    ])
    .with_columns(
        Flexibility_Performance_Index=(
            (pl.col("E_base_peak") - pl.col("E_actual_peak")) / 
            (pl.col("E_base_peak") + epsilon)
        ),
        High_Flexibility_Flag=(
            ((pl.col("E_base_peak") - pl.col("E_actual_peak")) / (pl.col("E_base_peak") + epsilon)) > 0.05
        ).cast(pl.Int32)
    )
)

# Merge FPI metrics back into the master experiment dataset so EVERY original column is preserved
final_ml_dataset = (
    df_with_baseline
    .filter(pl.col("Experiment_price_NOK_kWh").is_not_null())
    .join(fpi_aggregated, on=["ID", "Date"], how="inner")
)

print(f"Final ML-ready dataset generated with shape: {final_ml_dataset.shape}")
print(f"Total columns available for modeling: {len(final_ml_dataset.columns)}")

with pl.Config(tbl_cols=-1, tbl_width_chars=-1):
    print(final_ml_dataset)

Final ML-ready dataset generated with shape: (1460760, 30)
Total columns available for modeling: 30
shape: (1_460_760, 30)
┌────────┬─────────────────────────┬────────────┬─────────────┬────────────┬──────┬────────────┬───────────────┬─────────────┬──────────────────────────┬─────────────┬───────────────┬───────────────┬───────────────┬──────────────┬────────┬──────────────────────┬──────────────┬─────────┬─────────┬────────────────┬────────────────────────┬─────────────────────┬─────────────────┬─────────────────────┬──────────┬───────────────┬─────────────┬───────────────────────────────┬───────────────────────┐
│ ID     ┆ From                    ┆ Date       ┆ Day_Of_Week ┆ Is_Weekend ┆ Hour ┆ Demand_kWh ┆ Price_profile ┆ Price_level ┆ Experiment_price_NOK_kWh ┆ Temperature ┆ Temperature24 ┆ Temperature48 ┆ Temperature72 ┆ Municipality ┆ Region ┆ Control_Price_Phase2 ┆ Group_Phase2 ┆ Is_Zero ┆ Row_Idx ┆ Zero_Block_Key ┆ Consecutive_Zero_Hours ┆ Multi_Day_Zero_Flag ┆ Temperature_Bin 

In [42]:
what_percentage_missing(final_ml_dataset)

Percentage of missing values in each column:
shape: (30, 2)
┌───────────────────────────────┬─────────────┐
│ column                        ┆ pct_missing │
│ ---                           ┆ ---         │
│ str                           ┆ f64         │
╞═══════════════════════════════╪═════════════╡
│ Baseline_Demand_kWh           ┆ 100.0       │
│ Zero_Block_Key                ┆ 99.995482   │
│ Multi_Day_Zero_Flag           ┆ 99.995482   │
│ ID                            ┆ 0.0         │
│ From                          ┆ 0.0         │
│ Date                          ┆ 0.0         │
│ Day_Of_Week                   ┆ 0.0         │
│ Is_Weekend                    ┆ 0.0         │
│ Hour                          ┆ 0.0         │
│ Demand_kWh                    ┆ 0.0         │
│ Price_profile                 ┆ 0.0         │
│ Price_level                   ┆ 0.0         │
│ Experiment_price_NOK_kWh      ┆ 0.0         │
│ Temperature                   ┆ 0.0         │
│ Temperature24             